# Semantic Search & LLM Application
**Starter Notebook** dikerjakan oleh **Habibie Dzaky Lathif**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base -> Embedding -> Vector Database (FAISS)-> Semantic Search -> Prompt Design -> LLM Integration-> Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

---

**Catatan model.** Notebook ini menggunakan model yang lebih relevan, yaitu: **`gemini-embedding-2`** (embedding) dan **`gemini-3.6-flash`** (LLM). Cara pemanggilan API tetap sama dengan starter notebook.

**Sebelum menjalankan:** salin `.env.example` menjadi `.env` di root project, lalu isi `GEMINI_API_KEY`. Jalankan notebook dari awal sampai akhir (*Run All*).

## 1. Import Library & Load Knowledge Base

In [33]:
import os
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
import faiss
from google import genai  # library Gemini API: google-genai
from google.genai import types
from google.genai import errors
from dotenv import load_dotenv

# --- Path project: notebook bisa dijalankan dari folder notebook/ maupun dari root project
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'ecommerce_faq_knowledge_base.csv'
VECTOR_DIR = PROJECT_ROOT / 'vector_store'
INDEX_PATH = VECTOR_DIR / 'faq_index.faiss'
DOC_DIR = PROJECT_ROOT / 'documentation'
RESULTS_PATH = DOC_DIR / 'retrieval_test_results.md'

# --- Model (pengganti resmi model starter yang sudah shut down)
EMBEDDING_MODEL = 'gemini-embedding-2'   # menggantikan text-embedding-004
LLM_MODEL = 'gemini-3.6-flash'           # menggantikan gemini-2.0-flash
TOP_K = 3

# --- API key dibaca dari file .env (tidak ikut di-commit)
load_dotenv(PROJECT_ROOT / '.env')
api_key = os.environ.get('GEMINI_API_KEY')
assert api_key, 'GEMINI_API_KEY belum di-set. Salin .env.example menjadi .env di root project lalu isi API key Anda.'

client = genai.Client(api_key=api_key)

# --- Load knowledge base (file asli hanya dibaca, tidak pernah ditulis ulang)
df = pd.read_csv(DATA_PATH)
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


### 1.1 Pemeriksaan Struktur Data
Memeriksa jumlah entri, kolom yang tersedia, tipe data, distribusi kategori, nilai kosong, dan duplikasi sebelum data diproses.

In [34]:
print('Jumlah entri :', len(df))
print('Jumlah kolom :', df.shape[1])
print('Kolom        :', list(df.columns))
print()
print('Tipe data per kolom:')
print(df.dtypes.to_string())
print()
print('Nilai kosong per kolom:')
print(df.isna().sum().to_string())
print()
print('faq_id unik            :', df['faq_id'].is_unique)
print('Pertanyaan duplikat    :', df['question'].duplicated().sum())
print('Jumlah kategori        :', df['category'].nunique())

Jumlah entri : 59
Jumlah kolom : 4
Kolom        : ['faq_id', 'category', 'question', 'answer']

Tipe data per kolom:
faq_id      int64
category      str
question      str
answer        str

Nilai kosong per kolom:
faq_id      0
category    0
question    0
answer      0

faq_id unik            : True
Pertanyaan duplikat    : 0
Jumlah kategori        : 8


In [35]:
# Distribusi entri per kategori
category_counts = df['category'].value_counts().rename_axis('category').reset_index(name='jumlah_entri')
category_counts

,category,jumlah_entri
0,Pemesanan,9
1,Pengiriman,9
2,Pembayaran,8
3,Pengembalian,8
4,Akun,7
5,Produk,7
6,Promo,6
7,Customer Service,5


In [36]:
# Panjang teks (jumlah kata) untuk memastikan setiap entri cukup pendek sehingga tidak perlu chunking
length_stats = pd.DataFrame({
    'kata_question': df['question'].str.split().str.len(),
    'kata_answer': df['answer'].str.split().str.len(),
}).describe().round(1)
length_stats

,kata_question,kata_answer
count,59.0,59.0
mean,7.6,17.2
std,1.5,2.6
min,5.0,11.0
25%,6.5,15.0
50%,8.0,17.0
75%,9.0,19.0
max,11.0,23.0


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [37]:
# Kolom 'content' hanya ditambahkan pada DataFrame di memori; file CSV asli tidak diubah.
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Jumlah unit retrievable:', len(documents))
print('Contoh unit retrievable:')
print(documents[0])

Jumlah unit retrievable: 59
Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


**Alasan menggabungkan question + answer.** Pertanyaan pelanggan bisa mirip dengan *pertanyaan* FAQ (misalnya "cara batalin pesanan") atau dengan *isi jawaban* FAQ (misalnya "resi belum muncul" yang kata kuncinya ada di jawaban). Menggabungkan keduanya membuat satu vektor mewakili topik dan solusi sekaligus. Urutan `documents` sama dengan urutan baris `df`, sehingga posisi vektor di FAISS (0..n-1) dapat dipetakan kembali ke baris `df` dengan `df.iloc[idx]`.

## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [38]:
RETRYABLE_CODES = {429, 500, 503}  # rate limit / gangguan server sementara

def with_retry(fn, max_retries=5, base_wait=5):
    """Menjalankan fn(); jika kena rate limit/server error sementara, tunggu lalu coba lagi."""
    for attempt in range(max_retries):
        try:
            return fn()
        except errors.APIError as e:
            code = getattr(e, 'code', None)
            if code not in RETRYABLE_CODES or attempt == max_retries - 1:
                raise
            wait = base_wait * (2 ** attempt)
            print(f'  API error {code} (percobaan {attempt + 1}/{max_retries}), menunggu {wait} detik...')
            time.sleep(wait)


def get_embedding(text):
    result = with_retry(lambda: client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text
    ))
    return result.embeddings[0].values


start = time.time()
embeddings = [get_embedding(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print(f'Selesai membuat {len(embeddings)} embedding dalam {time.time() - start:.1f} detik')
print('Shape embeddings:', embeddings.shape)

Selesai membuat 59 embedding dalam 31.1 detik
Shape embeddings: (59, 3072)


### 3.1 Verifikasi Embedding
Sebelum disimpan, dipastikan: jumlah vektor sama dengan jumlah entri, semua vektor berdimensi sama, dan tidak ada nilai NaN/inf.

In [39]:
n_entries, dim = embeddings.shape
print('Jumlah vektor   :', n_entries, '| jumlah entri KB:', len(df))
print('Dimensi vektor  :', dim)
print('Tipe data       :', embeddings.dtype)
print('Ada NaN / inf?  :', bool(~np.isfinite(embeddings).all()))

norms = np.linalg.norm(embeddings, axis=1)
print(f'Norma L2 vektor : min={norms.min():.4f}, max={norms.max():.4f}')

assert n_entries == len(df), 'Jumlah embedding tidak sama dengan jumlah entri knowledge base!'
assert np.isfinite(embeddings).all(), 'Ada nilai NaN/inf pada embedding!'
print('\nVerifikasi embedding OK.')

Jumlah vektor   : 59 | jumlah entri KB: 59
Dimensi vektor  : 3072
Tipe data       : float32
Ada NaN / inf?  : False
Norma L2 vektor : min=1.0000, max=1.0000

Verifikasi embedding OK.


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

Index yang dipakai sama dengan starter: `IndexFlatL2` (exact search, cocok untuk 59 entri). Sebelum dimasukkan, setiap vektor dinormalisasi ke panjang 1 (`faiss.normalize_L2`). Dengan vektor ber-norma 1, jarak L2 kuadrat yang dikembalikan FAISS setara dengan cosine similarity: **cosine = 1 − distance / 2**. Jadi hasil pencarian bisa ditampilkan sebagai distance (sesuai FAISS) dan sebagai similarity score yang lebih mudah dibaca (rentang −1 sampai 1, makin tinggi makin mirip). Urutan peringkat hasil tidak berubah karena normalisasi ini.

In [40]:
faiss.normalize_L2(embeddings)  # normalisasi in-place

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)
assert index.ntotal == len(df), 'Jumlah vektor di index tidak sama dengan jumlah entri knowledge base!'

Jumlah vektor tersimpan di index: 59


In [41]:
# Simpan index ke disk agar vector database tersedia di repository
VECTOR_DIR.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(INDEX_PATH))

# Verifikasi: baca ulang index dari file
loaded_index = faiss.read_index(str(INDEX_PATH))
print('Index disimpan di      :', INDEX_PATH.relative_to(PROJECT_ROOT))
print('Ukuran file            :', f'{INDEX_PATH.stat().st_size / 1024:.1f} KB')
print('Vektor di index (file) :', loaded_index.ntotal, '| dimensi:', loaded_index.d)
assert loaded_index.ntotal == len(df) and loaded_index.d == dimension
print('Mapping posisi vektor -> entri: posisi i di index = baris i pada data/ecommerce_faq_knowledge_base.csv')

Index disimpan di      : vector_store\faq_index.faiss
Ukuran file            : 708.0 KB
Vektor di index (file) : 59 | dimensi: 3072
Mapping posisi vektor -> entri: posisi i di index = baris i pada data/ecommerce_faq_knowledge_base.csv


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [42]:
def semantic_search(query, top_k=TOP_K):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    faiss.normalize_L2(query_embedding)  # samakan perlakuan dengan vektor dokumen
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        distance = float(distances[0][rank])
        results.append({
            'rank': rank + 1,
            'faq_id': int(df.iloc[idx]['faq_id']),
            'category': df.iloc[idx]['category'],
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'distance': round(distance, 4),                       # L2 kuadrat dari FAISS (makin kecil makin mirip)
            'similarity': round(1 - distance / 2, 4),             # cosine similarity (makin besar makin mirip)
        })
    return results


def show_results(results):
    """Menampilkan hasil pencarian sebagai tabel."""
    return pd.DataFrame(results)[['rank', 'faq_id', 'category', 'question', 'similarity', 'distance']]


# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
show_results(test_results)

,rank,faq_id,category,question,similarity,distance
0,1,12,Pengiriman,Bagaimana jika paket saya belum sampai melebih...,0.6913,0.6173
1,2,14,Pengiriman,Kenapa status pengiriman saya tidak berubah se...,0.6906,0.6187
2,3,52,Pengiriman,Kenapa nomor resi saya belum bisa dilacak di w...,0.6827,0.6346


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

Desain prompt:

| Bagian | Isi | Tujuan |
|---|---|---|
| **Persona** | Asisten customer service e-commerce, ramah, ringkas, Bahasa Indonesia | Menjaga nada jawaban konsisten dengan kebutuhan tim CS |
| **Task** | Jawab **hanya** dari Context; abaikan FAQ yang tidak relevan; jangan menebak; jika info tidak ada, katakan belum tersedia dan sarankan menghubungi customer service | Mencegah LLM memakai pengetahuan umum (halusinasi) dan memberi perilaku yang jelas saat informasi tidak tersedia |
| **Context** | Top-k FAQ hasil semantic search, lengkap dengan nomor FAQ dan kategori | Grounding jawaban ke knowledge base dan memungkinkan sitasi sumber |
| **Format** | Maks. 4 kalimat, poin bernomor hanya untuk langkah-langkah, diakhiri baris `Sumber: FAQ #...` atau `Sumber: -` | Jawaban mudah dibaca pelanggan dan mudah diverifikasi saat pengujian |

In [43]:
def build_context(search_results):
    return '\n\n'.join(
        f"[FAQ #{r['faq_id']} | Kategori: {r['category']}]\n"
        f"Pertanyaan: {r['question']}\n"
        f"Jawaban: {r['answer']}"
        for r in search_results
    )


def build_prompt(query, search_results):
    context = build_context(search_results)
    prompt = f'''### PERSONA
Kamu adalah asisten customer service sebuah platform e-commerce di Indonesia. Kamu ramah, sopan, ringkas, dan menggunakan Bahasa Indonesia yang natural.

### TASK
Jawab pertanyaan pelanggan pada bagian PERTANYAAN PELANGGAN dengan aturan berikut:
1. Gunakan HANYA informasi yang ada di bagian CONTEXT. Jangan memakai pengetahuan umum, asumsi, atau kebijakan dari platform lain.
2. CONTEXT berisi beberapa FAQ hasil pencarian otomatis dan tidak semuanya pasti relevan. Gunakan hanya FAQ yang benar-benar menjawab pertanyaan, abaikan sisanya.
3. Jika CONTEXT tidak memuat informasi yang cukup untuk menjawab, jangan menebak. Sampaikan bahwa informasi tersebut belum tersedia di knowledge base FAQ, lalu sarankan pelanggan menghubungi customer service.
4. Jika hanya sebagian pertanyaan yang bisa dijawab dari CONTEXT, jawab bagian tersebut dan sebutkan bagian mana yang informasinya belum tersedia.

### CONTEXT
{context}

### FORMAT
- Gunakan Bahasa Indonesia dengan nada ramah dan profesional, sapa pelanggan dengan "Anda".
- Maksimal 4 kalimat. Gunakan poin bernomor hanya jika jawabannya berupa langkah-langkah.
- Akhiri jawaban dengan satu baris terpisah: "Sumber: FAQ #<nomor>" (pisahkan dengan koma jika lebih dari satu), atau "Sumber: -" jika informasi tidak tersedia di CONTEXT.

### PERTANYAAN PELANGGAN
{query}

### JAWABAN'''
    return prompt


# Contoh prompt yang terbentuk
print(build_prompt('kenapa paket saya belum sampai?', test_results))

### PERSONA
Kamu adalah asisten customer service sebuah platform e-commerce di Indonesia. Kamu ramah, sopan, ringkas, dan menggunakan Bahasa Indonesia yang natural.

### TASK
Jawab pertanyaan pelanggan pada bagian PERTANYAAN PELANGGAN dengan aturan berikut:
1. Gunakan HANYA informasi yang ada di bagian CONTEXT. Jangan memakai pengetahuan umum, asumsi, atau kebijakan dari platform lain.
2. CONTEXT berisi beberapa FAQ hasil pencarian otomatis dan tidak semuanya pasti relevan. Gunakan hanya FAQ yang benar-benar menjawab pertanyaan, abaikan sisanya.
3. Jika CONTEXT tidak memuat informasi yang cukup untuk menjawab, jangan menebak. Sampaikan bahwa informasi tersebut belum tersedia di knowledge base FAQ, lalu sarankan pelanggan menghubungi customer service.
4. Jika hanya sebagian pertanyaan yang bisa dijawab dari CONTEXT, jawab bagian tersebut dan sebutkan bagian mana yang informasinya belum tersedia.

### CONTEXT
[FAQ #12 | Kategori: Pengiriman]
Pertanyaan: Bagaimana jika paket saya belum sa

## 7. Mengintegrasikan LLM

In [44]:
def generate_answer(query, top_k=TOP_K):
    # 1) retrieve -> 2) susun prompt -> 3) generate
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = with_retry(lambda: client.models.generate_content(
        model=LLM_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3)
    ))
    answer = response.text if response.text else '(LLM tidak mengembalikan teks jawaban)'
    return answer.strip(), search_results


answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print(f"- FAQ #{s['faq_id']} (similarity {s['similarity']:.4f}): {s['question']}")

Jawaban LLM:
Halo! Ya, fitur cicilan tersedia untuk transaksi menggunakan kartu kredit tertentu pada nominal minimal yang ditentukan oleh pihak bank. 

Sumber: FAQ #20

Sumber FAQ yang digunakan:
- FAQ #20 (similarity 0.8086): Apakah saya bisa membayar dengan cicilan?
- FAQ #22 (similarity 0.7252): Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- FAQ #17 (similarity 0.7103): Metode pembayaran apa saja yang tersedia?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

**Rancangan query uji.** Setiap query diberi label `expected_faq_ids` (FAQ yang seharusnya terambil) yang ditentukan dengan membaca isi knowledge base sebelum pengujian. Dari label ini dihitung:
- **Hit@1**: FAQ yang benar berada di peringkat 1.
- **Hit@3**: FAQ yang benar berada di 3 hasil teratas (yang masuk ke Context LLM).

| # | Jenis | Query | FAQ yang diharapkan |
|---|---|---|---|
| Q1 | Jelas ada di Knowledge base | Metode pembayaran apa saja yang tersedia? | #17 |
| Q2 | Jelas ada di Knowledge base | Bagaimana cara membatalkan pesanan yang sudah dibuat? | #3 |
| Q3 | Parafrase / sinonim | uang saya sudah kepotong tapi orderan masih dibilang belum dibayar, gimana nih? | #21 |
| Q4 | Parafrase / sinonim | barang yang datang pecah waktu kardusnya saya buka, saya harus ngapain? | #16 |
| Q5 | Tidak ada di knowledge base | Apakah bisa kirim barang ke luar negeri? | – (tidak ada) |

Q1–Q2 memakai kalimat yang sama dengan pertanyaan FAQ. Q3–Q4 sengaja tidak memakai kata kunci FAQ (*dipotong → kepotong*, *status pesanan → orderan*, *belum terbayar → belum dibayar*, *kondisi rusak → pecah*, *paket diterima → kardus dibuka*). Q5 masih satu domain (pengiriman) tetapi knowledge base tidak membahas pengiriman internasional, sehingga menguji apakah sistem menolak menjawab walaupun FAQ pengiriman yang mirip ikut terambil.

In [45]:
test_queries = [
    {'id': 'Q1', 'type': 'Jelas ada di KB',
     'query': 'Metode pembayaran apa saja yang tersedia?',
     'expected_faq_ids': [17]},
    {'id': 'Q2', 'type': 'Jelas ada di KB',
     'query': 'Bagaimana cara membatalkan pesanan yang sudah dibuat?',
     'expected_faq_ids': [3]},
    {'id': 'Q3', 'type': 'Parafrase / sinonim',
     'query': 'uang saya sudah kepotong tapi orderan masih dibilang belum dibayar, gimana nih?',
     'expected_faq_ids': [21]},
    {'id': 'Q4', 'type': 'Parafrase / sinonim',
     'query': 'barang yang datang pecah waktu kardusnya saya buka, saya harus ngapain?',
     'expected_faq_ids': [16]},
    {'id': 'Q5', 'type': 'Tidak ada di KB',
     'query': 'Apakah bisa kirim barang ke luar negeri?',
     'expected_faq_ids': []},
]

test_records = []
for t in test_queries:
    q = t['query']
    print('=' * 100)
    print(f"{t['id']} [{t['type']}] Query: {q}")
    answer, sources = generate_answer(q)

    retrieved_ids = [s['faq_id'] for s in sources]
    print('\nHasil retrieval (top-3):')
    for s in sources:
        mark = '  <-- expected' if s['faq_id'] in t['expected_faq_ids'] else ''
        print(f"  {s['rank']}. FAQ #{s['faq_id']:<3} sim={s['similarity']:.4f} dist={s['distance']:.4f} | {s['question']}{mark}")
    print('\nJawaban LLM:')
    print(answer)
    print('Sumber teratas:', sources[0]['question'] if sources else '-')

    test_records.append({
        'id': t['id'],
        'type': t['type'],
        'query': q,
        'expected_faq_ids': t['expected_faq_ids'],
        'retrieved_faq_ids': retrieved_ids,
        'top1_similarity': sources[0]['similarity'],
        'hit@1': (retrieved_ids[0] in t['expected_faq_ids']) if t['expected_faq_ids'] else None,
        'hit@3': any(i in t['expected_faq_ids'] for i in retrieved_ids) if t['expected_faq_ids'] else None,
        'answer': answer,
        'sources': sources,
    })
print('=' * 100)

Q1 [Jelas ada di KB] Query: Metode pembayaran apa saja yang tersedia?
  API error 503 (percobaan 1/5), menunggu 5 detik...

Hasil retrieval (top-3):
  1. FAQ #17  sim=0.7817 dist=0.4365 | Metode pembayaran apa saja yang tersedia?  <-- expected
  2. FAQ #22  sim=0.7052 dist=0.5895 | Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
  3. FAQ #20  sim=0.7043 dist=0.5914 | Apakah saya bisa membayar dengan cicilan?

Jawaban LLM:
Halo! Untuk bertransaksi, Anda dapat memilih beberapa metode pembayaran yang tersedia, yaitu transfer bank, kartu kredit/debit, e-wallet, dan pembayaran melalui minimarket rekanan.

Sumber: FAQ #17
Sumber teratas: Metode pembayaran apa saja yang tersedia?
Q2 [Jelas ada di KB] Query: Bagaimana cara membatalkan pesanan yang sudah dibuat?

Hasil retrieval (top-3):
  1. FAQ #3   sim=0.7493 dist=0.5015 | Bagaimana cara membatalkan pesanan yang sudah dibuat?  <-- expected
  2. FAQ #44  sim=0.6605 dist=0.6790 | Bagaimana jika produk yang saya pesan ternyat

### 8.1 Ringkasan Hasil Pengujian

In [46]:
summary = pd.DataFrame(test_records)[
    ['id', 'type', 'query', 'expected_faq_ids', 'retrieved_faq_ids', 'top1_similarity', 'hit@1', 'hit@3']
]
summary

,id,type,query,expected_faq_ids,retrieved_faq_ids,top1_similarity,hit@1,hit@3
0,Q1,Jelas ada di KB,Metode pembayaran apa saja yang tersedia?,[17],"[17, 22, 20]",0.7817,True,True
1,Q2,Jelas ada di KB,Bagaimana cara membatalkan pesanan yang sudah ...,[3],"[3, 44, 1]",0.7493,True,True
2,Q3,Parafrase / sinonim,uang saya sudah kepotong tapi orderan masih di...,[21],"[21, 19, 8]",0.8288,True,True
3,Q4,Parafrase / sinonim,barang yang datang pecah waktu kardusnya saya ...,[16],"[16, 54, 24]",0.7234,True,True
4,Q5,Tidak ada di KB,Apakah bisa kirim barang ke luar negeri?,[],"[5, 15, 10]",0.6375,None,None


In [47]:
answerable = summary[summary['expected_faq_ids'].str.len() > 0]
print(f"Query yang punya jawaban di KB : {len(answerable)}")
print(f"Hit@1                          : {int(answerable['hit@1'].sum())}/{len(answerable)}")
print(f"Hit@3                          : {int(answerable['hit@3'].sum())}/{len(answerable)}")
print()
print('Similarity peringkat-1 per query:')
for r in test_records:
    print(f"  {r['id']} ({r['type']}): {r['top1_similarity']:.4f}")

Query yang punya jawaban di KB : 4
Hit@1                          : 4/4
Hit@3                          : 4/4

Similarity peringkat-1 per query:
  Q1 (Jelas ada di KB): 0.7817
  Q2 (Jelas ada di KB): 0.7493
  Q3 (Parafrase / sinonim): 0.8288
  Q4 (Parafrase / sinonim): 0.7234
  Q5 (Tidak ada di KB): 0.6375


### 8.2 Menyimpan Hasil Pengujian ke `documentation/retrieval_test_results.md`
Hasil mentah pengujian (retrieval top-3 beserta skor dan jawaban LLM) disimpan ke file markdown agar bisa dibaca di repository tanpa membuka notebook.

In [48]:
def md_escape(text):
    return str(text).replace('|', '\\|').replace('\n', '<br>')

lines = [
    '# Hasil Pengujian Retrieval & Response AI',
    '',
    f"_Dihasilkan otomatis oleh notebook pada {datetime.now().strftime('%Y-%m-%d %H:%M')}._",
    '',
    f'- Embedding model: `{EMBEDDING_MODEL}` (dimensi {dimension})',
    f'- LLM: `{LLM_MODEL}` (temperature 0.3)',
    f'- Vector database: FAISS `IndexFlatL2` dengan vektor ternormalisasi, {index.ntotal} vektor, top-k = {TOP_K}',
    '- Similarity = cosine similarity = 1 - distance/2 (makin tinggi makin mirip)',
    '',
    '## Ringkasan',
    '',
    '| ID | Jenis | Query | FAQ diharapkan | FAQ terambil (top-3) | Similarity #1 | Hit@1 | Hit@3 |',
    '|---|---|---|---|---|---|---|---|',
]
for r in test_records:
    exp = ', '.join(f'#{i}' for i in r['expected_faq_ids']) or '–'
    got = ', '.join(f'#{i}' for i in r['retrieved_faq_ids'])
    h1 = '–' if r['hit@1'] is None else ('Ya' if r['hit@1'] else 'Tidak')
    h3 = '–' if r['hit@3'] is None else ('Ya' if r['hit@3'] else 'Tidak')
    lines.append(f"| {r['id']} | {r['type']} | {md_escape(r['query'])} | {exp} | {got} | {r['top1_similarity']:.4f} | {h1} | {h3} |")

lines += ['', '## Detail per Query', '']
for r in test_records:
    lines += [
        f"### {r['id']} — {r['type']}",
        '',
        f"**Query:** {r['query']}",
        '',
        '| Rank | FAQ | Kategori | Pertanyaan FAQ | Similarity | Distance |',
        '|---|---|---|---|---|---|',
    ]
    for s in r['sources']:
        lines.append(f"| {s['rank']} | #{s['faq_id']} | {s['category']} | {md_escape(s['question'])} | {s['similarity']:.4f} | {s['distance']:.4f} |")
    lines += ['', '**Jawaban LLM:**', '', '> ' + r['answer'].replace('\n', '\n> '), '']

DOC_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.write_text('\n'.join(lines), encoding='utf-8')
print('Hasil pengujian disimpan di:', RESULTS_PATH.relative_to(PROJECT_ROOT))

Hasil pengujian disimpan di: documentation\retrieval_test_results.md


### Catatan Analisis Relevansi

Analisis ini berdasarkan output asli pengujian (lihat `documentation/retrieval_test_results.md`, dijalankan dengan `gemini-embedding-2` + `gemini-3.6-flash`).

| ID | Jenis | FAQ #1 terambil | Similarity #1 | Similarity #2 | Selisih #1–#2 | Hit@1 | Jawaban LLM |
|---|---|---|---|---|---|---|---|
| Q1 | Jelas ada di KB | #17 | 0.7817 | 0.7052 | 0.0765 | Ya | Tepat, sumber FAQ #17 |
| Q2 | Jelas ada di KB | #3 | 0.7493 | 0.6605 | 0.0888 | Ya | Tepat, sumber FAQ #3 |
| Q3 | Parafrase / sinonim | #21 | 0.8288 | 0.7199 | 0.1089 | Ya | Tepat, sumber FAQ #21 |
| Q4 | Parafrase / sinonim | #16 | 0.7234 | 0.6715 | 0.0519 | Ya | Tepat, sumber FAQ #16 |
| Q5 | Tidak ada di KB | #5 (tidak relevan) | 0.6375 | 0.6297 | 0.0078 | – | Menolak menjawab, `Sumber: -` |

**1. Query yang berhasil di-retrieve dengan baik.**
Keempat query yang jawabannya ada di knowledge base (Q1–Q4) mendapatkan FAQ yang benar di peringkat 1.Hasil terpenting ada di query parafrase:
- **Q3** ("uang saya sudah kepotong tapi orderan masih dibilang belum dibayar, gimana nih?") hampir tidak memakai kata kunci FAQ #21, tetapi justru mendapat similarity tertinggi dari semua query (0.8288) dan selisih terbesar dengan peringkat 2 (0.1089). Artinya model embedding menangkap padanan makna *kepotong ≈ dipotong*, *orderan ≈ pesanan*, *belum dibayar ≈ belum terbayar*, termasuk dalam bahasa informal.
- **Q4** ("barang yang datang pecah waktu kardusnya saya buka") juga berhasil menemukan FAQ #16 ("paket diterima dalam kondisi rusak") walaupun tidak ada kata *rusak* maupun *paket* di query. Ini menunjukkan pencarian bekerja berdasarkan makna, bukan pencocokan kata kunci.
- Menariknya, Q1 dan Q2 yang kalimatnya **identik** dengan pertanyaan FAQ tidak mendapat skor mendekati 1 (hanya 0.78 dan 0.75). Penyebabnya, vektor dokumen dibuat dari gabungan `Q: ... A: ...`, sehingga isi jawaban ikut menggeser vektor dokumen menjauh dari teks pertanyaannya saja. Skor absolut karena itu tidak bisa dibaca sebagai "persentase kecocokan"; yang lebih bermakna adalah urutan peringkat dan selisih antar peringkat.

**2. Query dan hasil yang kurang relevan.**
- **Q4 adalah kasus paling tipis** di antara query yang bisa dijawab: similarity #1 terendah (0.7234) dan selisih dengan peringkat 2 hanya 0.0519. Peringkat 2–3 adalah FAQ #54 (menerima barang yang salah) dan #24 (prosedur pengembalian barang tidak sesuai). Keduanya masih satu tema (komplain/pengembalian), jadi wajar ikut terambil, tetapi aturannya berbeda (FAQ #24 memberi batas 3 hari, FAQ #16 memberi batas 2x24 jam). Jika urutan sampai tertukar, LLM berisiko memberikan prosedur yang salah. Ini menunjukkan FAQ yang topiknya berdekatan rawan saling tertukar.
- **Peringkat 2–3 tidak selalu relevan.** Pada Q2 (membatalkan pesanan), FAQ #44 (stok habis) masuk peringkat 2 karena jawabannya mengandung kata *membatalkan pesanan*, dan FAQ #1 (cara memesan) masuk peringkat 3 karena sama-sama membahas pesanan. Pada Q1, FAQ #22 dan #20 masih berkaitan (pembayaran gabungan dan cicilan). Karena top-k selalu 3, Context yang dikirim ke LLM hampir selalu membawa 1–2 FAQ yang tidak dibutuhkan; ini sesuai dengan keterbatasan #1 dan #3 di README.
- **Q5 (pengiriman ke luar negeri)** tidak punya jawaban di KB, tetapi retrieval tetap mengembalikan 3 FAQ pengiriman/pemesanan (#5 ubah alamat, #15 pilih jasa kirim, #10 lacak paket). Similarity #1 Q5 (0.6375) jelas lebih rendah dari semua query yang bisa dijawab (terendah 0.7234), dan ketiga hasilnya hampir sama skornya (selisih #1–#2 hanya 0.0078). Pola "skor rendah dan datar" ini adalah tanda bahwa tidak ada FAQ yang benar-benar cocok. Pada sampel kecil ini, ambang sekitar 0,68–0,70 akan memisahkan Q5 dari Q1–Q4, tetapi angka itu perlu divalidasi dengan lebih banyak query sebelum dipakai.

**3. Respons LLM saat informasi tersedia.**
Semua jawaban untuk Q1–Q4 sesuai isi FAQ, tidak menambah informasi dari luar knowledge base, dan mencantumkan satu sumber yang benar. LLM juga berhasil **mengabaikan FAQ yang tidak relevan** di Context: misalnya pada Q4 jawaban hanya memakai aturan 2x24 jam dari FAQ #16 dan tidak mencampurnya dengan batas 3 hari dari FAQ #24, dan pada Q2 tidak menyinggung kasus stok habis. Format juga dipatuhi: jawaban ringkas, poin bernomor hanya dipakai di Q4 karena isinya berupa langkah, dan setiap jawaban diakhiri baris `Sumber`.

**4. Respons LLM saat informasi tidak tersedia (Q5).**
LLM menyatakan bahwa informasi pengiriman ke luar negeri belum tersedia di knowledge base, menyarankan pelanggan menghubungi customer service, dan menutup dengan `Sumber: -`. LLM tidak tergoda menyimpulkan jawaban dari FAQ yang mirip, misalnya mengubah FAQ #15 ("dapat memilih jasa pengiriman") menjadi "bisa, silakan pilih kurir internasional". Ini menunjukkan instruksi grounding di prompt (Task poin 1 dan 3) efektif. Namun perlu dicatat bahwa penolakan ini **sepenuhnya bergantung pada kepatuhan LLM**, karena retrieval sendiri tetap mengirim 3 FAQ ke Context tanpa penyaringan.

**5. Kesimpulan dan rekomendasi.**
Pada 5 query uji, pipeline semantic search + LLM bekerja sesuai tujuan: retrieval akurat untuk query baku maupun parafrase yang terlihat seperti kalimat informal atau tak baku, jawaban LLM berdasar dan dapat ditelusuri ke sumbernya, dan sistem mengakui ketidaktahuannya untuk pertanyaan di luar knowledge base. Jumlah query uji masih kecil, jadi hasil ini belum cukup untuk menyimpulkan akurasi secara umum. Perbaikan yang disarankan:
1. Menambahkan **ambang similarity** (dikalibrasi dengan lebih banyak query) agar query di luar KB bisa langsung dialihkan ke customer service tanpa bergantung pada LLM, dan FAQ dengan skor rendah tidak ikut masuk Context.
2. Menambah jumlah dan variasi query uji, terutama pertanyaan di antara topik yang berdekatan (rusak vs salah kirim vs pengembalian) dan pertanyaan gabungan yang butuh lebih dari satu FAQ.
3. Mencoba embedding **pertanyaan saja** atau memakai *task type* retrieval yang berbeda untuk query dan dokumen, lalu membandingkan skor dan selisih antar peringkatnya dengan pendekatan Q + A saat ini.